# 02 · 채점 — "맞았다"를 숫자로

시스템의 답이 정답과 글자까지 똑같은 경우는 드물다.
그래서 **부분 점수 규칙**이 필요하다. MemoryOS 논문은 LoCoMo에서 두 가지를 쓴다:

> "On the LoCoMo benchmark, standard **F1** and **BLEU-1** are employed" — 논문 원문

이 노트북에서는 둘을 손으로 계산해 본다. 함수는 `memlab.evaluation`에 있다.

## F1 — 겹침 점수

채점은 3단계다: ① 소문자로 ② 문장부호 떼고 단어로 쪼개고 ③ 겹침을 잰다.

In [1]:
from memlab.evaluation import bleu1, set_f1, standard_f1, tokenize

정답 = "mental health"
시스템답 = "It raised awareness for mental health issues."

정답_단어 = set(tokenize(정답))
답_단어 = set(tokenize(시스템답))
겹침 = 답_단어 & 정답_단어

print(f"정답 단어:   {정답_단어}")
print(f"시스템 단어: {답_단어}")
print(f"겹침:        {겹침}\n")

p = len(겹침) / len(답_단어)   # 말한 것 중 정답 단어 비율
r = len(겹침) / len(정답_단어)  # 정답 단어 중 말한 비율
print(f"precision = {len(겹침)}/{len(답_단어)} = {p:.2f}   (군더더기 벌점)")
print(f"recall    = {len(겹침)}/{len(정답_단어)} = {r:.2f}   (빠뜨림 벌점)")
print(f"F1 = 2pr/(p+r) = {2 * p * r / (p + r):.2f}")
print(f"set_f1()으로 확인 → {set_f1(시스템답, 정답):.2f}")

정답 단어:   {'health', 'mental'}
시스템 단어: {'raised', 'issues', 'for', 'health', 'mental', 'it', 'awareness'}
겹침:        {'health', 'mental'}

precision = 2/7 = 0.29   (군더더기 벌점)
recall    = 2/2 = 1.00   (빠뜨림 벌점)
F1 = 2pr/(p+r) = 0.44
set_f1()으로 확인 → 0.44


**내용은 맞았는데 0.44점.** 길게 말한 만큼 precision이 깎였다.
F1은 "정답 단어만, 군더더기 없이"를 보상한다 —
벤치마크의 답변 프롬프트가 늘 "간결하게 답하라"고 하는 이유다.

## 원본 코드의 F1은 조금 다르다

논문은 "standard F1"이라고 썼지만, 원본 repo 코드는 단어를 **set에 넣는다** —
같은 단어의 반복이 무시된다. 반복이 있는 답에서만 두 방식이 갈린다:

In [2]:
답 = "health health health"
print(f"원본 방식 set_f1:  {set_f1(답, 정답):.2f}   ← 반복을 못 봄")
print(f"표준 standard_f1:  {standard_f1(답, 정답):.2f}   ← 반복을 벌함")

원본 방식 set_f1:  0.67   ← 반복을 못 봄
표준 standard_f1:  0.40   ← 반복을 벌함


그래서 우리 채점기는 **둘 다** 계산한다.
원본과 비교할 땐 `set_f1`, 다른 논문과 비교할 땐 `standard_f1`.

## BLEU-1 — 정밀도 + 짧은 답 벌점

BLEU-1은 recall이 없다. 벌점이 없다면 한 단어만 맞혀도 만점이다.
그래서 정답보다 짧은 답은 exp(1 − 정답길이/답길이)로 깎는다:

In [3]:
print(f"bleu1('mental health')  = {bleu1('mental health', 정답):.2f}")
print(f"bleu1('mental')         = {bleu1('mental', 정답):.2f}   ← 정밀도 1.0이지만 벌점 e^-1")
print(f"bleu1(긴 답)            = {bleu1(시스템답, 정답):.2f}   ← 벌점은 없지만 정밀도 2/7")

bleu1('mental health')  = 1.00
bleu1('mental')         = 0.37   ← 정밀도 1.0이지만 벌점 e^-1
bleu1(긴 답)            = 0.29   ← 벌점은 없지만 정밀도 2/7


## cat5(adversarial)는 채점이 뒤집힌다

cat5는 정답이 없다("모른다"가 정답). 그런데 원본 실험 코드는 결과를 저장할 때
**정답 칸에 함정 오답(adversarial_answer)을 넣는다.** 그 상태로 F1을 계산하면:

- 시스템이 잘 버티면 ("I don't know") → 함정 답과 안 겹침 → **0점**
- 시스템이 속아서 지어내면 → 함정 답과 겹침 → **높은 점수**

즉 **cat5는 F1이 높을수록 나쁘다** (환각을 잰 것).
cat1~4와 절대 한 평균에 섞으면 안 되고, 우리 채점 리포트는 cat5를 따로 표시한다.

## 어휘 지표의 한계

F1/BLEU는 **단어가 겹치는지**만 본다. 뜻은 모른다:

In [4]:
# 뜻은 같은데 감점
print(f"'May 7th, 2023' vs '7 May 2023'  → F1 {set_f1('May 7th, 2023', '7 May 2023'):.2f}")
# 뜻은 틀렸는데 득점
print(f"'not mental health' vs 'mental health' → F1 {set_f1('not mental health', 'mental health'):.2f}")

'May 7th, 2023' vs '7 May 2023'  → F1 0.67
'not mental health' vs 'mental health' → F1 0.80


같은 날짜인데 `7th ≠ 7`이라 감점되고, 뜻이 정반대인 답(`not ...`)이 고득점한다.

이 약점 때문에 최근 논문들(Mem0, LongMemEval 등)은 **LLM-as-a-Judge**
(LLM에게 "이 답 맞아?"를 묻기)를 병용한다. 대신 비용이 들고, judge 판정이
흔들려서 재현성이 나빠진다.

**우리 선택**: MemoryOS 논문과 같은 자(F1/BLEU-1)로 baseline을 재고,
내 메소드 실험 때 필요하면 judge를 추가한다. 두 가지 실용 교훈:

1. 답변 프롬프트의 형식 지시(날짜 표기 등)가 점수를 흔든다 → **원본 프롬프트 그대로**
2. 지표가 단어 겹침이므로, 점수 차이를 해석할 땐 실제 답을 몇 개 눈으로 볼 것

## 정리

- **F1** = 군더더기 벌점(precision) × 빠뜨림 벌점(recall)의 조화평균
- 원본 코드는 set 방식(반복 무시) — 논문의 "standard F1"과 다르다 → 둘 다 계산
- **BLEU-1** = 정밀도 + 짧은 답 벌점
- cat5는 함정 답 기준이라 **높을수록 나쁨** — 따로 본다
- 채점 함수가 원본과 같다는 건 차분 테스트가 보증한다 (`tests/test_metrics.py`)

**다음 — 03 · MemoryOS 해부**: 학생(memory 시스템)이 실제로 뭘 하는지 —
저장하고, 버리고, 찾아서 답하는 구조를 원본 코드로 읽는다.